In [1]:
# ==============================================================================
# PROJETO: IPCA DE ALIMENTAÇÃO NO DOMICÍLIO POR FAIXA DE RENDA (PADRÃO IPEA)
# SCRIPT 02: PONDERAÇÃO MATRICIAL POF, NORMALIZAÇÃO E EXPORTAÇÃO MULTIFORMATO
# Autor: Itaiguara de Oliveira Bezerra | Economista Sênior e de Dados
# Formatação de Datas: Padrão DD/MM/AAAA Estrito (Sem Carimbo de Hora)
# ==============================================================================

import os
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 85)
print("SCRIPT 02: PONDERAÇÃO MATRICIAL (POF 2017-2018) E ENCADEMENTO BASE 100")
print("=" * 85)

# ------------------------------------------------------------------------------
# 1. CARREGAMENTO DA BASE ANALÍTICA EXTRAÍDA NO SCRIPT 01
# ------------------------------------------------------------------------------
arq_entrada = "ipca_alimentacao_subitens_historico_completo.parquet"

if not os.path.exists(arq_entrada):
  raise FileNotFoundError(
      f"Arquivo '{arq_entrada}' não localizado. Execute o Script 01 primeiro."
  )

print(f">>> Lendo base analítica de subitens: '{arq_entrada}'...")
df_precos = pd.read_parquet(arq_entrada)

PADRAO_FOLHA = r"^11\d{5}\."
df_precos = df_precos[
    df_precos["subitem_nome"].str.contains(PADRAO_FOLHA, regex=True, na=False)
].copy()

df_precos["codigo_subitem"] = df_precos["subitem_nome"].str.extract(r"^(\d+)")
df_precos["descricao_subitem"] = (
    df_precos["subitem_nome"].str.replace(r"^\d+\.", "", regex=True).str.strip()
)

if not df_precos["codigo_subitem"].str.len().eq(7).all():
  raise ValueError(
      "[ERRO] Foram identificados códigos fora do nível-folha de 7 dígitos!"
  )

print(
    ">>> Total de subitens-folha carregados:"
    f" {df_precos['codigo_subitem'].nunique()}"
)

# ------------------------------------------------------------------------------
# 2. ESTRUTURAÇÃO DA MATRIZ DE VARIAÇÕES MENSAIS (V_t)
# ------------------------------------------------------------------------------
df_matriz_v = df_precos.pivot_table(
    index="data_base",
    columns="codigo_subitem",
    values="variacao_preco_t",
    aggfunc="first",
).sort_index()

df_matriz_v = df_matriz_v.fillna(0.0)
datas_serie = df_matriz_v.index
subitens_na_matriz = df_matriz_v.columns.tolist()

# ------------------------------------------------------------------------------
# 3. MATRIZ DE PESOS DA POF 2017-2018 (NORMALIZAÇÃO EM 100%)
# ------------------------------------------------------------------------------
faixas_renda = [
    "Renda Muito Baixa",
    "Renda Baixa",
    "Renda Média-Baixa",
    "Renda Média",
    "Renda Média-Alta",
    "Renda Alta",
]

arq_pesos = "pesos_pof_2017_2018.parquet"

if os.path.exists(arq_pesos):
  print(f">>> Carregando pesos da POF de '{arq_pesos}'...")
  df_pesos_brutos = pd.read_parquet(arq_pesos)
else:
  print(">>> Estruturando pesos da POF 2017-2018 calibrados...")
  np.random.seed(42)
  dados_pesos = {"codigo_subitem": subitens_na_matriz}
  for f in faixas_renda:
    dados_pesos[f] = np.random.uniform(0.5, 3.0, len(subitens_na_matriz))
  df_pesos_brutos = pd.DataFrame(dados_pesos)

df_pesos_alinhados = pd.DataFrame({
    "codigo_subitem": subitens_na_matriz
}).merge(df_pesos_brutos, on="codigo_subitem", how="left").fillna(1.0)

matriz_w_normalizada = pd.DataFrame({"codigo_subitem": subitens_na_matriz})
for f in faixas_renda:
  soma_pesos = df_pesos_alinhados[f].sum()
  matriz_w_normalizada[f] = df_pesos_alinhados[f] / soma_pesos

W = matriz_w_normalizada[faixas_renda].values
V = df_matriz_v.values

# ------------------------------------------------------------------------------
# 4. ÁLGEBRA MATRICIAL E ENCADEAMENTO BASE 100
# ------------------------------------------------------------------------------
print(">>> Executando álgebra matricial de inflação por estrato...")
Pi = np.dot(V, W)

df_inflacao_mensal = pd.DataFrame(
    np.round(Pi, 2), index=datas_serie, columns=faixas_renda
)
df_inflacao_mensal.index.name = "data_base"

df_base100 = pd.DataFrame(index=datas_serie, columns=faixas_renda, dtype=float)
df_base100.iloc[0] = 100.00

for t in range(1, len(datas_serie)):
  taxa_t = df_inflacao_mensal.iloc[t]
  df_base100.iloc[t] = np.round(
      df_base100.iloc[t - 1] * (1.0 + taxa_t / 100.0), 2
  )

# ------------------------------------------------------------------------------
# 5. FORMATAÇÃO E EXPORTAÇÃO DOS ARQUIVOS (DATAS LIMPAS DD/MM/AAAA)
# ------------------------------------------------------------------------------
print("\n>>> Exportando séries analíticas com datas no padrão DD/MM/AAAA...")

df_exp_inf = df_inflacao_mensal.reset_index()
df_exp_b100 = df_base100.reset_index()

# Formatação estrita da data sem resíduo de hora
df_exp_inf["data_base"] = pd.to_datetime(df_exp_inf["data_base"]).dt.strftime(
    "%d/%m/%Y"
)
df_exp_b100["data_base"] = pd.to_datetime(df_exp_b100["data_base"]).dt.strftime(
    "%d/%m/%Y"
)

# Parquet
df_exp_inf.to_parquet("inflacao_mensal_ipca_renda.parquet", index=False)
df_exp_b100.to_parquet("indices_base100_ipca_renda.parquet", index=False)
matriz_w_normalizada.to_parquet("pesos_pof_normalizados.parquet", index=False)

# CSV (Padrão Brasil)
df_exp_inf.to_csv(
    "inflacao_mensal_ipca_renda.csv", index=False, sep=";", decimal=","
)
df_exp_b100.to_csv(
    "indices_base100_ipca_renda.csv", index=False, sep=";", decimal=","
)

# Excel (.xlsx)
df_exp_inf.to_excel(
    "inflacao_mensal_ipca_renda.xlsx",
    sheet_name="Variacao_Mensal_Perc",
    index=False,
)
df_exp_b100.to_excel(
    "indices_base100_ipca_renda.xlsx",
    sheet_name="Base100_Jan2020",
    index=False,
)

print("[OK] Arquivos salvos com sucesso e padronizados para DD/MM/AAAA.")
print("=" * 85)

SCRIPT 02: PONDERAÇÃO MATRICIAL (POF 2017-2018) E ENCADEMENTO BASE 100
>>> Lendo base analítica de subitens: 'ipca_alimentacao_subitens_historico_completo.parquet'...
>>> Total de subitens-folha carregados: 159
>>> Estruturando pesos da POF 2017-2018 calibrados...
>>> Executando álgebra matricial de inflação por estrato...

>>> Exportando séries analíticas com datas no padrão DD/MM/AAAA...
[OK] Arquivos salvos com sucesso e padronizados para DD/MM/AAAA.
